In [2]:
import pandas as pd
import numpy as np
from torchvision.transforms import transforms
import torch
from sklearn.model_selection import train_test_split

In [3]:
import torch.optim as optim
# load the datasets
train = pd.read_csv("fashion-mnist_train.csv")
test = pd.read_csv("fashion-mnist_test.csv")

print(train.shape)
print(test.shape)

(60000, 785)
(10000, 785)


In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [10]:
X_train = train.drop('label' , axis=1)
y_train = train['label']
X_test = test.drop('label' , axis=1)
y_test = test['label']

In [16]:
X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

In [17]:
custom_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean = [.485 , .456 , .406] , std = [.229 , .224 , .225])
])

In [43]:
# create dataset and dataloader
from torch.utils.data import Dataset , DataLoader
from PIL import Image

class CustomDataset(Dataset):
    def __init__(self,features,labels ,transform):
        self.features = features
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.features)

    def __getitem__(self,idx):

        # change the shape to 28*28
        img = self.features[idx].reshape(28,28)
        # change the dtype to uint8
        img = img.astype(np.uint8)

        # change black&white to color
        img = np.stack([img]*3 , axis = -1) # this gives us (H,W,C) if axis=-1 otherwise (C,H,W)

        # change into PIL image
        img = Image.fromarray(img)

        #Apply transformations
        img = self.transform(img)

        return img , torch.tensor(self.labels[idx] , dtype = torch.long)

In [44]:
train_dataset = CustomDataset(X_train , y_train , custom_transform)
test_dataset = CustomDataset(X_test , y_test , custom_transform)
train_loader = DataLoader(train_dataset , batch_size=512 , shuffle=True,pin_memory=True)
test_loader = DataLoader(test_dataset , batch_size=512 , shuffle=True,pin_memory=True)

In [45]:
# Let us fetch the pretrained model from torchvision
import torchvision.models as models

vgg16 = models.vgg16(pretrained = True)

d:\Study\Deep_Learning\Pytorch_Course\.venv\Lib\site-packages\torchvision\models\_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\Study\Deep_Learning\Pytorch_Course\.venv\Lib\site-packages\torchvision\models\_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [46]:
vgg16

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [47]:
# Freeze the parameters of features layer
for params in vgg16.features.parameters():
    params.requires_grad = False

In [48]:
import torch.nn as nn

In [49]:
# Let us build our own classifier

vgg16.classifier = nn.Sequential(
    nn.Linear(25088 , 1024),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(1024 , 512),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(512 , 10)

)

In [50]:
vgg16

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [51]:
vgg16 = vgg16.to(device)

In [52]:
learning_rate = 0.0001
epochs = 10

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vgg16.classifier.parameters() , lr = learning_rate)

In [53]:
# Let us now build the back propagation code
import time
start_time = time.time()
for epoch in range(epochs):
    total_loss = 0
    for batch_features,batch_labels in train_loader:
        # save them in GPU
        batch_features,batch_labels = batch_features.to(device), batch_labels.to(device)

        # forward pass

        outputs = vgg16(batch_features)

        # calc loss

        loss = criterion(outputs , batch_labels) # this automatically converts logits to probs using softmax

        # remove the gradients
        optimizer.zero_grad()

        # back propagate
        loss.backward()

        # update the weights
        optimizer.step()

        # find the sum of loss in each epoch
        total_loss += loss
    print(f'Loss at epoch {epoch+1} ->{total_loss/len(train_loader)}')
print(time.time()-start_time)

OutOfMemoryError: CUDA out of memory. Tried to allocate 6.12 GiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Of the allocated memory 6.78 GiB is allocated by PyTorch, and 22.45 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# Evaluation 
# set to eval mode
vgg16.eval()
total = 0
correct = 0
for batch_features , batch_labels in test_loader:
    batch_features , batch_labels  = batch_features.to(device), batch_labels.to(device)

    # forward pass
    outputs = vgg16(batch_features)

    # get the max values of outputs
    _,predicted = torch.max(outputs , 1)
    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()

print(f'Accuracy Score -> {correct/total}')


# Evaluation of training data
# set to eval mode
vgg16.eval()
total = 0
correct = 0
for batch_features , batch_labels in train_loader:
    batch_features , batch_labels  = batch_features.to(device), batch_labels.to(device)

    # forward pass
    outputs = vgg16(batch_features)

    # get the max values of outputs
    _,predicted = torch.max(outputs , 1)
    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()

print(f'Accuracy Score -> {correct/total}')

torch.Size([512, 1, 28, 28])
torch.Size([512])
Loss at epoch 1 ->0.7426297664642334
Loss at epoch 2 ->0.42303231358528137
Loss at epoch 3 ->0.3605176508426666
Loss at epoch 4 ->0.32145074009895325
Loss at epoch 5 ->0.2984146475791931
Loss at epoch 6 ->0.27817606925964355
Loss at epoch 7 ->0.2606474459171295
Loss at epoch 8 ->0.24676062166690826
Loss at epoch 9 ->0.23283255100250244
Loss at epoch 10 ->0.22074559330940247
Loss at epoch 11 ->0.20736974477767944
Loss at epoch 12 ->0.20400434732437134
Loss at epoch 13 ->0.19392693042755127
Loss at epoch 14 ->0.1873529553413391
Loss at epoch 15 ->0.17834866046905518
Loss at epoch 16 ->0.16754424571990967
Loss at epoch 17 ->0.16442646086215973
Loss at epoch 18 ->0.15602706372737885
Loss at epoch 19 ->0.1510400027036667
Loss at epoch 20 ->0.1418970227241516
Loss at epoch 21 ->0.13415177166461945
Loss at epoch 22 ->0.13509109616279602
Loss at epoch 23 ->0.12571175396442413
Loss at epoch 24 ->0.1215876117348671
Loss at epoch 25 ->0.1194201782345